# TEMPO Tutorial: End-to-End Example with Real Data

This tutorial walks through running TEMPO on **real single-cell RNA-seq data** from the paper ([Auerbach et al., *Nature Communications* 2022](https://www.nature.com/articles/s41467-022-34185-w)). We use smooth muscle cells (SMCs) from the mouse aorta.

By the end of this tutorial, you will:
1. Load and inspect the scRNA-seq data
2. Configure and run TEMPO
3. Load and interpret cell phase estimates
4. Visualize circadian phase distributions
5. Identify de novo cycling genes
6. Interpret the Bayes factor for clock evidence

## 1. Download the Data

The SMC aorta data from the paper is available on Dropbox. Download the `adata.h5ad` file from the shared folder and place it in a local directory:

**Dropbox link:** https://www.dropbox.com/sh/tl0ty163vyg265i/AAApt14eybExMMPK7VVDmfvga

Navigate to `smc/adata.h5ad` and download it. Then set the path below to where you saved the file.

In [ ]:
import os

# --- SET THE PATH TO THE DOWNLOADED ADATA FILE ---
# Update this path to where you saved the downloaded adata.h5ad file
adata_path = "smc_adata.h5ad"

if not os.path.exists(adata_path):
    raise FileNotFoundError(
        f"Could not find '{adata_path}'.\n"
        "Please download the SMC adata.h5ad file from:\n"
        "https://www.dropbox.com/sh/tl0ty163vyg265i/AAApt14eybExMMPK7VVDmfvga\n"
        "(navigate to smc/adata.h5ad)\n"
        "Then update the 'adata_path' variable above to point to the downloaded file."
    )
print(f"Found data file: {adata_path}")

## 2. Inspect the Data

Let's load the AnnData object and look at its structure. TEMPO expects raw (unnormalized) transcript counts in a [cells x genes] matrix.

In [ ]:
import anndata

adata = anndata.read_h5ad(adata_path)

print(f"Number of cells: {adata.n_obs}")
print(f"Number of genes: {adata.n_vars}")
print(f"\nFirst 10 gene names: {adata.var_names[:10].tolist()}")
print(f"\nCell metadata columns: {adata.obs.columns.tolist()}")
print(f"\nadata.X (count matrix):\n{adata.X[:5, :5]}")

## 3. Select Reference Files

TEMPO needs two reference files:
1. **Core clock gene list** — a text file listing known circadian clock genes (one per line)
2. **Gene acrophase prior** — a CSV specifying the expected peak expression time (acrophase) for each clock gene in radians

Since we are working with mouse aorta data, we use the shipped aorta-specific reference files. The acrophase priors were derived from bulk circadian transcriptomics studies of the aorta, so they reflect the tissue-specific expression timing of clock genes.

In [ ]:
import pandas as pd

# --- PATHS TO SHIPPED REFERENCE FILES ---
# (paths are relative to the repo root; adjust if running from a different directory)
core_clock_gene_path = "../data/core_clock_genes.txt"
gene_acrophase_prior_path = "../data/clock_aorta_acrophase_prior.csv"
reference_gene = "Arntl"

# --- VIEW CORE CLOCK GENES ---
with open(core_clock_gene_path) as f:
    clock_genes = [line.strip() for line in f if line.strip()]
print(f"Core clock genes ({len(clock_genes)} total):")
print(clock_genes)

# Check which clock genes are present in the data
shared_genes = [g for g in clock_genes if g in adata.var_names]
print(f"\nClock genes found in dataset: {len(shared_genes)} / {len(clock_genes)}")
print(shared_genes)

In [ ]:
import numpy as np

# --- VIEW ACROPHASE PRIORS ---
acrophase_df = pd.read_csv(gene_acrophase_prior_path, index_col="gene")
print("Gene acrophase priors (aorta-specific):\n")

# Add a column showing the acrophase in circadian hours for easier interpretation
acrophase_df["peak_hour (CT)"] = acrophase_df["prior_acrophase_loc"] * 24 / (2 * np.pi)
print(acrophase_df.to_string())

## 4. Configure and Run TEMPO

We now configure and run TEMPO using the Python API. The key settings:
- `reference_gene = "Arntl"`: anchors the phase so that phase 0 corresponds to peak Arntl expression
- `use_nb = True`: use negative binomial likelihood (recommended for scRNA-seq count data)
- `num_phase_grid_points = 24`: 24 grid points = 1-hour resolution for the phase posterior
- `max_num_alg_steps = 2`: run Steps 1 and 2 up to 2 times

All other parameters use their defaults. See the README for the full parameter reference.

In [ ]:
import tempo
from tempo import unsupervised_alg

# --- SET OUTPUT FOLDER ---
folder_out = "smc_tempo_results"

# --- RUN TEMPO ---
tempo.unsupervised_alg.run(
    adata=adata,
    folder_out=folder_out,
    gene_acrophase_prior_path=gene_acrophase_prior_path,
    core_clock_gene_path=core_clock_gene_path,
    reference_gene=reference_gene,
    min_gene_prop=1e-5,
    use_nb=True,
    num_phase_grid_points=24,
    max_num_alg_steps=2,
    vi_max_epochs=300
)

## 5. Explore Cell Phase Outputs

The main cell-level output is `tempo_results/opt/cell_posterior.tsv`. Each row is a cell, and the columns (`bin_0` through `bin_23`) are the posterior density at each of the 24 phase grid points, evenly spaced from 0 to 2*pi.

We can use TEMPO's built-in `ThetaPosteriorDist` class to work with these posteriors.

In [ ]:
import torch
from tempo import cell_posterior

# --- LOAD CELL PHASE POSTERIORS ---
cell_posterior_path = f"{folder_out}/tempo_results/opt/cell_posterior.tsv"
cell_posterior_df = pd.read_table(cell_posterior_path, sep="\t", index_col="barcode")

print(f"Cell posterior shape: {cell_posterior_df.shape}")
print(f"Number of cells: {cell_posterior_df.shape[0]}")
print(f"Number of phase bins: {cell_posterior_df.shape[1]}")
cell_posterior_df.head()

In [ ]:
# --- CREATE POSTERIOR DISTRIBUTION OBJECT ---
cell_posterior_obj = cell_posterior.ThetaPosteriorDist(
    torch.Tensor(np.array(cell_posterior_df))
)

# --- GET MAP (MAXIMUM A POSTERIORI) PHASE FOR EACH CELL ---
map_phases = cell_posterior_obj.map_phase.numpy()

# Convert to circadian hours (0 to 24)
map_hours = map_phases * 24 / (2 * np.pi)

print(f"MAP phases (radians), first 10 cells: {map_phases[:10]}")
print(f"MAP phases (CT hours), first 10 cells: {np.round(map_hours[:10], 1)}")

In [ ]:
# --- COMPUTE 95% CREDIBLE INTERVALS ---
confidence_intervals = cell_posterior_obj.compute_confidence_interval(confidence=0.95)

# Interval sizes in hours (each bin = 1 hour when using 24 grid points)
interval_sizes = np.sum(confidence_intervals, axis=1)

print(f"Mean 95% credible interval size: {np.mean(interval_sizes):.1f} hours")
print(f"Median 95% credible interval size: {np.median(interval_sizes):.1f} hours")
print(f"Cells with interval < 12 hours (confident): {np.sum(interval_sizes < 12)} / {len(interval_sizes)}")

## 6. Visualize Results

### 6a. Posterior density for a single cell

Each cell has a full posterior distribution over phase. Cells with strong circadian signal will have a peaked distribution; uncertain cells will have a flatter distribution.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

# --- PLOT POSTERIOR FOR A SINGLE CELL ---
cell_index = 0
phase_grid = cell_posterior_obj.phase_grid.numpy()
phase_grid_hours = phase_grid * 24 / (2 * np.pi)
density = cell_posterior_obj.theta_posterior_likelihood[cell_index, :].numpy()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(phase_grid_hours, density, width=0.8, color="steelblue", alpha=0.8)
ax.set_xlabel("Circadian Phase (CT hours)", fontsize=12)
ax.set_ylabel("Posterior Density", fontsize=12)
ax.set_title(f"Phase Posterior for Cell: {cell_posterior_df.index[cell_index]}", fontsize=13)
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 3))
plt.tight_layout()
plt.show()

### 6b. Polar histogram of all cell phases

A polar plot shows the distribution of MAP cell phases across the circadian cycle. Phase 0 (top) corresponds to peak Arntl expression.

In [ ]:
# --- POLAR HISTOGRAM OF CELL PHASES ---
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': 'polar'})

# Histogram
n_bins = 48
ax.hist(map_phases, bins=n_bins, density=True, color="steelblue", alpha=0.7, edgecolor="white")

# Formatting
ax.set_theta_zero_location("N")  # 0 at top
ax.set_theta_direction(-1)       # clockwise

# Hour labels
hour_ticks = np.linspace(0, 2 * np.pi, 24, endpoint=False)
ax.set_xticks(hour_ticks)
ax.set_xticklabels([f"CT{h}" for h in range(24)], fontsize=8)
ax.set_title("Distribution of Cell Circadian Phases\n(MAP estimates)", fontsize=13, pad=20)

plt.tight_layout()
plt.show()

### 6c. Distribution of credible interval sizes

This histogram shows how certain TEMPO is about each cell's phase. Smaller intervals mean more confident estimates.

In [ ]:
# --- HISTOGRAM OF CREDIBLE INTERVAL SIZES ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(interval_sizes, bins=range(0, 26), color="coral", alpha=0.8, edgecolor="white")
ax.axvline(x=12, color="black", linestyle="--", label="12-hour threshold")
ax.set_xlabel("95% Credible Interval Size (hours)", fontsize=12)
ax.set_ylabel("Number of Cells", fontsize=12)
ax.set_title("Cell Phase Uncertainty", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Identify De Novo Cycling Genes

TEMPO identifies genes that cycle beyond the core clock genes provided as input. These "de novo cyclers" are genes whose expression shows a significant circadian pattern based on the estimated cell phases.

The cycling gene results are in `cycler_gene_prior_and_posterior.tsv`. Key columns:
- **`phi_loc`**: posterior acrophase (peak expression phase) in radians
- **`A_loc`**: posterior amplitude (strength of oscillation)
- **`Q_prob_loc`**: posterior probability that the gene has non-zero amplitude (cycling probability)

In [ ]:
# --- LOAD CYCLING GENE RESULTS ---
cycler_path = f"{folder_out}/tempo_results/opt/cycler_gene_prior_and_posterior.tsv"
gene_df = pd.read_table(cycler_path, sep="\t", index_col="gene")

# Add interpretable columns
gene_df["peak_hour"] = gene_df["phi_loc"] * 24 / (2 * np.pi)
gene_df["is_core_clock"] = gene_df.index.isin(clock_genes)

print(f"Total cycling genes identified: {len(gene_df)}")
print(f"Core clock genes: {gene_df['is_core_clock'].sum()}")
print(f"De novo cyclers: {(~gene_df['is_core_clock']).sum()}")

# Show key columns, sorted by amplitude
gene_df[["A_loc", "phi_loc", "peak_hour", "Q_prob_loc", "is_core_clock"]].sort_values(
    "A_loc", ascending=False
).head(20)

### Gene acrophase polar plot

This plot shows the peak expression phase (acrophase) and amplitude for each cycling gene on a polar plot. The angle represents when the gene peaks, and the distance from center represents the amplitude.

In [ ]:
# --- GENE ACROPHASE POLAR PLOT ---
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})

# Plot core clock genes and de novo cyclers separately
clock_mask = gene_df["is_core_clock"]

ax.scatter(gene_df.loc[clock_mask, "phi_loc"], gene_df.loc[clock_mask, "A_loc"],
           c="red", s=80, label="Core clock", zorder=3, edgecolors="darkred")
ax.scatter(gene_df.loc[~clock_mask, "phi_loc"], gene_df.loc[~clock_mask, "A_loc"],
           c="dodgerblue", s=50, label="De novo cycler", zorder=2, alpha=0.7, edgecolors="navy")

# Label the core clock genes
for gene_name in gene_df.index[clock_mask]:
    ax.annotate(gene_name,
                xy=(gene_df.loc[gene_name, "phi_loc"], gene_df.loc[gene_name, "A_loc"]),
                fontsize=7, ha="left", va="bottom")

# Formatting
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
hour_ticks = np.linspace(0, 2 * np.pi, 24, endpoint=False)
ax.set_xticks(hour_ticks)
ax.set_xticklabels([f"CT{h}" for h in range(24)], fontsize=8)
ax.set_title("Gene Acrophases and Amplitudes", fontsize=13, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.show()

## 8. Interpret the Bayes Factor

TEMPO computes a Bayes factor at each iteration comparing the evidence for circadian clock structure vs. a null model (random gene-phase assignments). This is stored in `evidence/step_{p}_clock_evidence.txt`.

A higher log10 Bayes factor indicates stronger evidence for circadian signal:
- **< 0**: evidence favors the null (no detectable clock signal)
- **0 to 0.5**: weak evidence
- **0.5 to 1**: moderate evidence
- **> 1**: strong evidence

By default, TEMPO requires a log10 Bayes factor > log10(1.5) ~ 0.176 to continue iterating.

In [ ]:
# --- LOAD AND DISPLAY BAYES FACTOR EVIDENCE ---
evidence_dir = f"{folder_out}/evidence"

# Load null evidence
with open(f"{evidence_dir}/null_log_evidence_vec.txt") as f:
    null_evidence = float(f.read().strip())
print(f"Null (random) log10 evidence: {null_evidence:.4f}")

# Load evidence for each step
step = 1
while os.path.exists(f"{evidence_dir}/step_{step}_clock_evidence.txt"):
    with open(f"{evidence_dir}/step_{step}_clock_evidence.txt") as f:
        step_evidence = float(f.read().strip())
    bf = step_evidence - null_evidence
    print(f"Step {step} log10 evidence: {step_evidence:.4f}  |  Log10 Bayes factor vs null: {bf:.4f}")
    step += 1

print(f"\nThreshold for continuing: log10(1.5) = {np.log10(1.5):.4f}")

## Summary

In this tutorial, we:
1. **Loaded** real mouse aorta SMC scRNA-seq data
2. **Selected** the appropriate tissue-specific reference files (core clock genes and aorta acrophase priors)
3. **Ran TEMPO** to estimate cell circadian phases and identify de novo cycling genes
4. **Visualized** individual cell posteriors, the population-level phase distribution, and gene acrophases
5. **Interpreted** the Bayes factor to confirm circadian signal in the data

For more details on parameters and outputs, see:
- `tutorial_tempo_inputs.ipynb` — full parameter reference
- `tutorial_tempo_outputs.ipynb` — detailed output file descriptions
- The [README](../README.md) — comprehensive documentation